## Delegating Work with Handoffs

# Agentic Patterns: The Handoff Pattern

## Introduction & Overview

Welcome to another lesson about agentic patterns! In the previous lesson, you mastered orchestrating agents as tools, where a central planner agent could dynamically delegate tasks to specialist agents and receive their results back. Today, we're exploring a fundamentally different approach called the **handoff pattern**, where agents can completely transfer control to other specialized agents rather than just calling them as tools.

In this lesson, you'll extend the `Agent` class constructor to support handoff targets, create a handoff tool schema that enables control transfers, and implement the core handoff logic that cleanly passes conversation context between agents. We'll build a practical example with a general assistant that can hand off mathematical problems to a specialized calculator assistant, demonstrating how agents make intelligent decisions about when to transfer control versus handling tasks themselves.

## Understanding the Handoff Pattern

The handoff pattern represents a different philosophy of agent collaboration compared to the tool delegation approach you learned previously.

* **Tool Delegation:** The agent asks for help while maintaining responsibility for the final response. The user interacts only with the orchestrator.
* **Handoff Pattern:** The original agent says, "this other agent is better equipped to handle this entire conversation from here on." The specialist agent then responds directly to the user, and the original agent is no longer involved.

This pattern is particularly powerful when you have agents with very different capabilities or when the nature of a request clearly falls into one agent's domain of expertise.

## Extending the Agent Class Constructor

To implement handoffs, we need to extend our existing `Agent` class with the ability to track available handoff targets.

```python
class Agent:
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        handoffs=None,  # New parameter for handoff targets
        max_turns=10
    ):
        self.client = anthropic.Anthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns
        
        # Avoid shared mutable defaults
        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        self.handoffs = [] if handoffs is None else list(handoffs)  # List of agents for handoffs

```

## Creating the Handoff Tool Schema

We need a tool schema that allows the agent to request handoffs. This schema is automatically added to the agent's available tools when handoff targets are provided.

```python
# Define handoff tool schema
self.handoff_schema = {
    "name": "handoff",
    "description": "Transfer control to another specialized agent. Use this when the user's request is better handled by a different agent.",
    "input_schema": {
        "type": "object",
        "properties": {
            "name": {
                "type": "string",
                "description": f"Name of the agent to handoff to. Available agents: {[agent.name for agent in self.handoffs]}"
            },
            "reason": {
                "type": "string", 
                "description": "Brief explanation of why this handoff is needed"
            }
        },
        "required": ["name", "reason"]
    }
}

```

## Implementing the Handoff Logic

The `call_handoff` method handles the transfer of control. It performs several critical steps:

1. **Parameter Extraction:** Gets the target name and reason.
2. **Agent Lookup:** Finds the target agent in the `handoffs` list.
3. **Context Cleaning:** Removes the assistant's message containing the handoff call so the target agent receives a clean history.
4. **Control Transfer:** Calls the target agent's `run` method.

## When to Use Agents as Tools vs Handoffs

* **Use Agents as Tools:** When you need an orchestrator to maintain control and synthesize multiple inputs into a unified response (e.g., planning a trip).
* **Use Handoffs:** When a specialist is better equipped to handle the *entire* conversation from a certain point forward (e.g., passing a general query to a dedicated math or coding specialist).

## Summary & Best Practices

* **Define Clear Criteria:** Use prompts to help agents decide exactly when to transfer.
* **Context Management:** Always clean the conversation history before the handoff.
* **Robust Error Handling:** Ensure your system can recover if a handoff target is not found or fails.
* **Avoid Circularity:** Design handoff chains with clear directionality to prevent agents from passing control back and forth indefinitely.

## Building the Handoff Infrastructure

It's time to build the foundation that will enable agents to transfer control to one another! In this exercise, you'll extend the Agent class to support handoffs by adding some of the necessary infrastructure.

    Modify the Agent class constructor in src/agent.py:
        Add a handoffs parameter (default to None)
        Initialize self.handoffs as an empty list if None, otherwise copy the provided list

    Create the self.handoff_schema dictionary that defines the handoff tool:
        Set tool name to "handoff"
        Add a description explaining it transfers control to specialized agents
        Define input_schema with two required properties:
            "name": target agent's name
            "reason": why the handoff is needed
        Use a list comprehension over self.handoffs to dynamically list available agents in the "name" property's description

    Update src/main.py:
        Add handoffs=[calculator_assistant] to the helpful_assistant agent
        Print the handoff_schema attribute to verify the implementation

The displayed schema will show exactly how your agent will see the handoff tool when deciding whether to use it.

```
# agent.py
import json
import anthropic


class Agent:
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    # TODO: Add the handoffs parameter to the constructor (default to None)
    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        max_turns=10
    ):
        self.client = anthropic.Anthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns

        # Avoid shared mutable defaults and protect against external mutation
        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        # TODO: Initialize self.handoffs as an empty list if None, otherwise create a copy of the provided list
        
        # TODO: Define self.handoff_schema as a dictionary with the handoff tool schema
        # The schema should have:
        # - "name": "handoff"
        # - "description": explaining it transfers control to specialized agents
        # - "input_schema" with "name" and "reason" properties (both required)
        # - The "name" property description should dynamically list available agents from self.handoffs
        
    def _extract_text(self, content):
        return "".join(
            block.text for block in content
            if getattr(block, "type", None) == "text"
        )    

    def _build_request_args(self, messages):
        request_args = {
            "model": self.model,
            "system": self.system_prompt,
            "messages": messages,
            "max_tokens": 8000,
        }

        # Add tool schemas only if they exist
        if self.tool_schemas:
            request_args["tools"] = self.tool_schemas

        return request_args

    def call_tool(self, tool_use):
        tool_name = tool_use.name
        tool_input = tool_use.input or {}
        tool_use_id = tool_use.id

        print(f"🔧 Tool called: {tool_name}({tool_input})")
        
        try:
            result = str(self.tools[tool_name](**tool_input))
        except KeyError:
            result = f"Error: Tool {tool_name} not found"
        except Exception as e:
            result = f"Error: {str(e)}"

        return {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result
        }

    def run(self, input_messages):
        # Create a copy of the input messages to avoid modifying the original
        messages = input_messages.copy()

        # Loop until the model returns a final answer or the max turns is reached
        turn = 0
        while turn < self.max_turns:
            turn += 1

            response = self.client.messages.create(**self._build_request_args(messages))

            messages.append({"role": "assistant", "content": response.content})

            # Execute all tools if Claude requests any
            if response.stop_reason == "tool_use":
                tool_results = []
                for content_item in response.content:
                    if content_item.type == "tool_use":
                        tool_result = self.call_tool(content_item)
                        tool_results.append(tool_result)

                # Add all tool results to messages
                messages.append({
                    "role": "user",
                    "content": tool_results
                })
        
            else:
                # Return if no tools are requested
                response_text = self._extract_text(response.content)

                return messages, response_text

        # If the max turns is reached, raise an exception
        raise Exception("Max turns reached")


# main.py

import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load tool schemas
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Math tools
math_tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}

# Create a calculator assistant
calculator_assistant = Agent(
    name="calculator_assistant",
    system_prompt="You are a calculator assistant. You specialize in mathematical calculations and solving equations.",
    tools=math_tools,
    tool_schemas=tool_schemas
)

# TODO: Add the handoffs parameter to helpful_assistant, setting it to [calculator_assistant]
helpful_assistant = Agent(
    name="helpful_assistant",
    system_prompt="You are a helpful assistant. You can assist with various tasks and handoff to the calculator assistant for math problems."
)

# TODO: Print the handoff_schema attribute from helpful_assistant to verify it was created
# Hint: Use json.dumps with indent=2 for pretty printing
```

Here is the updated code for `agent.py` and `main.py` with the handoff infrastructure implemented as requested.

### 1. Updated `agent.py`

I have added the `handoffs` parameter to the `__init__` method, initialized the handoff list, and created the `self.handoff_schema` definition.

```python
import json
import anthropic

class Agent:
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        handoffs=None,  # Added handoffs parameter
        max_turns=10
    ):
        self.client = anthropic.Anthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns

        # Avoid shared mutable defaults and protect against external mutation
        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        
        # Initialize handoffs list
        self.handoffs = [] if handoffs is None else list(handoffs)

        # Define handoff tool schema
        self.handoff_schema = {
            "name": "handoff",
            "description": "Transfer control to another specialized agent. Use this when the user's request is better handled by a different agent.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": f"Name of the agent to handoff to. Available agents: {[agent.name for agent in self.handoffs]}"
                    },
                    "reason": {
                        "type": "string",
                        "description": "Brief explanation of why this handoff is needed"
                    }
                },
                "required": ["name", "reason"]
            }
        }

    # ... (rest of the Agent class methods remain the same)

```

---

### 2. Updated `main.py`

I have updated the `helpful_assistant` instantiation to include the `handoffs` parameter and added the print statement to verify the generated schema.

```python
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load tool schemas
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Math tools
math_tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}

# Create a calculator assistant
calculator_assistant = Agent(
    name="calculator_assistant",
    system_prompt="You are a calculator assistant. You specialize in mathematical calculations and solving equations.",
    tools=math_tools,
    tool_schemas=tool_schemas
)

# Add handoffs to helpful_assistant
helpful_assistant = Agent(
    name="helpful_assistant",
    system_prompt="You are a helpful assistant. You can assist with various tasks and handoff to the calculator assistant for math problems.",
    handoffs=[calculator_assistant]
)

# Print the handoff_schema attribute to verify
print("=== Handoff Schema ===")
print(json.dumps(helpful_assistant.handoff_schema, indent=2))

```

When you run this updated `main.py`, you will see a JSON printout of the `handoff` tool schema, confirming that the list of available agents (`['calculator_assistant']`) has been dynamically injected into the tool description.

## Combining Tool Schemas for Agent Requests

Great work setting up the handoff infrastructure in the previous exercise! Now it's time to make those handoff tools actually available to your agents by updating how request arguments are built.

Your task is to modify the _build_request_args method so that it builds a complete tools list that combines both regular tool schemas and the handoff schema.

    In the _build_request_args method:
        Create an empty list called all_tools at the start of the tool-building logic.
        If self.tool_schemas exists:
            Use extend() to add self.tool_schemas to all_tools.
        If self.handoffs is not empty:
            Use append() to add self.handoff_schema to all_tools.
        Only add all_tools to request_args["tools"] if the list is not empty.

    In main.py, test your implementation:
        Create a simple test message and store it in a list called messages.
        Call _build_request_args() directly on the helpful_assistant and store the result in request_args.
        Use json.dumps() with indent=2 to pretty-print the request arguments.

This verification step is crucial because it lets you see exactly what Claude will receive when deciding which tools to use!

```
# agent.py
import json
import anthropic


class Agent:
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        handoffs=None,
        max_turns=10
    ):
        self.client = anthropic.Anthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns

        # Avoid shared mutable defaults and protect against external mutation
        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        self.handoffs = [] if handoffs is None else list(handoffs)
        
        # Define handoff tool schema
        self.handoff_schema = {
            "name": "handoff",
            "description": "Transfer control to another specialized agent. Use this when the user's request is better handled by a different agent.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": f"Name of the agent to handoff to. Available agents: {[agent.name for agent in self.handoffs]}"
                    },
                    "reason": {
                        "type": "string", 
                        "description": "Brief explanation of why this handoff is needed"
                    }
                },
                "required": ["name", "reason"]
            }
        }
        
    def _extract_text(self, content):
        return "".join(
            block.text for block in content
            if getattr(block, "type", None) == "text"
        )    

    # TODO: Modify this method to build a complete tools list that includes both regular tool schemas and the handoff schema
    # Steps:
    # 1. Create an empty list called all_tools
    # 2. Use extend() to add self.tool_schemas to all_tools if self.tool_schemas exists
    # 3. Use append() to add self.handoff_schema to all_tools if self.handoffs is not empty
    # 4. Only add all_tools to request_args["tools"] if all_tools is not empty
    def _build_request_args(self, messages):
        request_args = {
            "model": self.model,
            "system": self.system_prompt,
            "messages": messages,
            "max_tokens": 8000,
        }

        # Add tool schemas only if they exist
        if self.tool_schemas:
            request_args["tools"] = self.tool_schemas

        return request_args

    def call_tool(self, tool_use):
        tool_name = tool_use.name
        tool_input = tool_use.input or {}
        tool_use_id = tool_use.id

        print(f"🔧 Tool called: {tool_name}({tool_input})")
        
        try:
            result = str(self.tools[tool_name](**tool_input))
        except KeyError:
            result = f"Error: Tool {tool_name} not found"
        except Exception as e:
            result = f"Error: {str(e)}"

        return {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result
        }

    def run(self, input_messages):
        # Create a copy of the input messages to avoid modifying the original
        messages = input_messages.copy()

        # Loop until the model returns a final answer or the max turns is reached
        turn = 0
        while turn < self.max_turns:
            turn += 1

            response = self.client.messages.create(**self._build_request_args(messages))

            messages.append({"role": "assistant", "content": response.content})

            # Execute all tools if Claude requests any
            if response.stop_reason == "tool_use":
                tool_results = []
                for content_item in response.content:
                    if content_item.type == "tool_use":
                        tool_result = self.call_tool(content_item)
                        tool_results.append(tool_result)

                # Add all tool results to messages
                messages.append({
                    "role": "user",
                    "content": tool_results
                })
        
            else:
                # Return if no tools are requested
                response_text = self._extract_text(response.content)

                return messages, response_text

        # If the max turns is reached, raise an exception
        raise Exception("Max turns reached")

# main.py
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load tool schemas
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Math tools
math_tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}

# Create a calculator assistant with math tools
calculator_assistant = Agent(
    name="calculator_assistant",
    system_prompt="You are a calculator assistant. You specialize in mathematical calculations and solving equations.",
    tools=math_tools,
    tool_schemas=tool_schemas
)

# Create a general assistant that can handoff to calculator
helpful_assistant = Agent(
    name="helpful_assistant",
    system_prompt="You are a helpful assistant. You can assist with various tasks and handoff to the calculator assistant for math problems.",
    handoffs=[calculator_assistant]
)

# TODO: Create a simple test message list with a user message (content can be anything simple like "Can you help me?")

# TODO: Call helpful_assistant._build_request_args(messages) and store the result in a variable called request_args

# TODO: Print the request_args using json.dumps(request_args, indent=2) to inspect the tools array
```

Here is the complete, working code for both files (`agent.py` and `main.py`) with all placeholders fully implemented according to your requirements.

### File: `agent.py`

```python
import json
import anthropic


class Agent:
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        handoffs=None,
        max_turns=10
    ):
        self.client = anthropic.Anthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns

        # Avoid shared mutable defaults and protect against external mutation
        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        self.handoffs = [] if handoffs is None else list(handoffs)
        
        # Define handoff tool schema
        self.handoff_schema = {
            "name": "handoff",
            "description": "Transfer control to another specialized agent. Use this when the user's request is better handled by a different agent.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": f"Name of the agent to handoff to. Available agents: {[agent.name for agent in self.handoffs]}"
                    },
                    "reason": {
                        "type": "string", 
                        "description": "Brief explanation of why this handoff is needed"
                    }
                },
                "required": ["name", "reason"]
            }
        }
        
    def _extract_text(self, content):
        return "".join(
            block.text for block in content
            if getattr(block, "type", None) == "text"
        )    

    def _build_request_args(self, messages):
        request_args = {
            "model": self.model,
            "system": self.system_prompt,
            "messages": messages,
            "max_tokens": 8000,
        }

        # 1. Create an empty list called all_tools
        all_tools = []

        # 2. Use extend() to add self.tool_schemas to all_tools if self.tool_schemas exists
        if self.tool_schemas:
            all_tools.extend(self.tool_schemas)

        # 3. Use append() to add self.handoff_schema to all_tools if self.handoffs is not empty
        if self.handoffs:
            all_tools.append(self.handoff_schema)

        # 4. Only add all_tools to request_args["tools"] if all_tools is not empty
        if all_tools:
            request_args["tools"] = all_tools

        return request_args

    def call_tool(self, tool_use):
        tool_name = tool_use.name
        tool_input = tool_use.input or {}
        tool_use_id = tool_use.id

        print(f"🔧 Tool called: {tool_name}({tool_input})")
        
        try:
            result = str(self.tools[tool_name](**tool_input))
        except KeyError:
            result = f"Error: Tool {tool_name} not found"
        except Exception as e:
            result = f"Error: {str(e)}"

        return {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result
        }

    def run(self, input_messages):
        # Create a copy of the input messages to avoid modifying the original
        messages = input_messages.copy()

        # Loop until the model returns a final answer or the max turns is reached
        turn = 0
        while turn < self.max_turns:
            turn += 1

            response = self.client.messages.create(**self._build_request_args(messages))

            messages.append({"role": "assistant", "content": response.content})

            # Execute all tools if Claude requests any
            if response.stop_reason == "tool_use":
                tool_results = []
                for content_item in response.content:
                    if content_item.type == "tool_use":
                        tool_result = self.call_tool(content_item)
                        tool_results.append(tool_result)

                # Add all tool results to messages
                messages.append({
                    "role": "user",
                    "content": tool_results
                })
        
            else:
                # Return if no tools are requested
                response_text = self._extract_text(response.content)

                return messages, response_text

        # If the max turns is reached, raise an exception
        raise Exception("Max turns reached")

```

---

### File: `main.py`

```python
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load tool schemas
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Math tools
math_tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}

# Create a calculator assistant with math tools
calculator_assistant = Agent(
    name="calculator_assistant",
    system_prompt="You are a calculator assistant. You specialize in mathematical calculations and solving equations.",
    tools=math_tools,
    tool_schemas=tool_schemas
)

# Create a general assistant that can handoff to calculator
helpful_assistant = Agent(
    name="helpful_assistant",
    system_prompt="You are a helpful assistant. You can assist with various tasks and handoff to the calculator assistant for math problems.",
    handoffs=[calculator_assistant]
)

# TODO: Create a simple test message list with a user message (content can be anything simple like "Can you help me?")
messages = [
    {
        "role": "user",
        "content": "Can you help me calculate the hypotenuse of a triangle?"
    }
]

# TODO: Call helpful_assistant._build_request_args(messages) and store the result in a variable called request_args
request_args = helpful_assistant._build_request_args(messages)

# TODO: Print the request_args using json.dumps(request_args, indent=2) to inspect the tools array
print("================================================================================")
print("INSPECTING CLAUDE REQUEST ARGUMENTS PAYLOAD")
print("================================================================================")
print(json.dumps(request_args, indent=2))
print("================================================================================")

```

## Implementing the Core Handoff Logic

Now that your agents can advertise handoff capabilities, it's time to implement the mechanism that finds target agents, cleans conversation context, and executes the control transfer.

Implement the call_handoff method in src/agent.py to:

    Extract agent name and reason from tool_use input and print the target agent name and handoff reason
    Use next() with a generator expression to find the target agent in self.handoffs
    Remove the last assistant message from messages (hides internal handoff mechanics)
    Call target_agent.run(clean_messages) and return (True, result) on success
    Handle StopIteration (agent not found) and general exceptions, returning (False, tool_result_dict) with appropriate error messages

In src/main.py, call the call_handoff() method using the provided MockToolUse class and messages. Test both a successful handoff to calculator_assistant and a failed handoff to a non-existent agent, printing the results to verify the handoff mechanics work correctly!

```
# agent.py
import json
import anthropic


class Agent:
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        handoffs=None,
        max_turns=10
    ):
        self.client = anthropic.Anthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns

        # Avoid shared mutable defaults and protect against external mutation
        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        self.handoffs = [] if handoffs is None else list(handoffs)
        
        # Define handoff tool schema
        self.handoff_schema = {
            "name": "handoff",
            "description": "Transfer control to another specialized agent. Use this when the user's request is better handled by a different agent.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": f"Name of the agent to handoff to. Available agents: {[agent.name for agent in self.handoffs]}"
                    },
                    "reason": {
                        "type": "string", 
                        "description": "Brief explanation of why this handoff is needed"
                    }
                },
                "required": ["name", "reason"]
            }
        }
        
    def _extract_text(self, content):
        return "".join(
            block.text for block in content
            if getattr(block, "type", None) == "text"
        )    

    def _build_request_args(self, messages):
        request_args = {
            "model": self.model,
            "system": self.system_prompt,
            "messages": messages,
            "max_tokens": 8000,
        }

        # Build the complete tool schemas list
        all_tools = []
        
        # Add regular tool schemas if they exist
        if self.tool_schemas:
            all_tools.extend(self.tool_schemas)
        
        # Add handoff schema if handoffs are available
        if self.handoffs:
            all_tools.append(self.handoff_schema)
        
        # Add tools to request if any exist
        if all_tools:
            request_args["tools"] = all_tools

        return request_args

    def call_tool(self, tool_use):
        tool_name = tool_use.name
        tool_input = tool_use.input or {}
        tool_use_id = tool_use.id

        print(f"🔧 Tool called: {tool_name}({tool_input})")
        
        try:
            result = str(self.tools[tool_name](**tool_input))
        except KeyError:
            result = f"Error: Tool {tool_name} not found"
        except Exception as e:
            result = f"Error: {str(e)}"

        return {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result
        }
    
    def call_handoff(self, tool_use, messages):
        # TODO: Extract the agent name from tool_use.input using .get("name")
        # TODO: Extract the reason from tool_use.input using .get("reason", "No reason provided")
        # TODO: Print a debug message with the target agent name and handoff reason
        
        # TODO: Start a try-except block to handle potential errors
            # TODO: Use next() with a generator expression to find the target agent in self.handoffs where agent.name matches agent_name
            # TODO: Clean the messages by removing the last assistant message if it exists 
            # Hint: Use conditional list slicing: messages[:-1] if messages and messages[-1]["role"] == "assistant" else messages
            # TODO: Call target_agent.run(clean_messages) and return a tuple: (True, result)
            
        # TODO: Add except StopIteration block to catch when agent is not found
            # TODO: Return (False, tool_result_dict) with an error message
            
        # TODO: Add except Exception as e block to catch any other errors
            # TODO: Return (False, tool_result_dict) with an error message

    def run(self, input_messages):
        # Create a copy of the input messages to avoid modifying the original
        messages = input_messages.copy()

        # Loop until the model returns a final answer or the max turns is reached
        turn = 0
        while turn < self.max_turns:
            turn += 1

            response = self.client.messages.create(**self._build_request_args(messages))

            messages.append({"role": "assistant", "content": response.content})

            # Execute all tools if Claude requests any
            if response.stop_reason == "tool_use":
                tool_results = []
                for content_item in response.content:
                    if content_item.type == "tool_use":
                        tool_result = self.call_tool(content_item)
                        tool_results.append(tool_result)

                # Add all tool results to messages
                messages.append({
                    "role": "user",
                    "content": tool_results
                })
        
            else:
                # Return if no tools are requested
                response_text = self._extract_text(response.content)

                return messages, response_text

        # If the max turns is reached, raise an exception
        raise Exception("Max turns reached")

# main.py
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load tool schemas
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Math tools
math_tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}

# Create a calculator assistant
calculator_assistant = Agent(
    name="calculator_assistant",
    system_prompt="You are a calculator assistant. You specialize in mathematical calculations and solving equations.",
    tools=math_tools,
    tool_schemas=tool_schemas
)

# Create a general assistant (can handoff to calculator)
helpful_assistant = Agent(
    name="helpful_assistant",
    system_prompt="You are a helpful assistant. You can assist with various tasks and handoff to the calculator assistant for math problems.",
    handoffs=[calculator_assistant]
)

# Mock class to simulate a tool_use object
class MockToolUse:
    def __init__(self, name, input_dict, id):
        self.name = name
        self.input = input_dict
        self.id = id

# Test case 1: Valid handoff to calculator_assistant
print("=== Test 1: Valid Handoff ===")
mock_tool_use = MockToolUse(
    name="handoff",
    input_dict={"name": "calculator_assistant", "reason": "Testing handoff logic"},
    id="test_handoff_1"
)

# Test messages with user and assistant messages
messages = [
    {"role": "user", "content": "Can you solve 2 + 2?"},
    {"role": "assistant", "content": [{"type": "text", "text": "Let me handoff to the calculator."}]}
]

# TODO: Call helpful_assistant.call_handoff(mock_tool_use, messages) and store the result in two variables: success and result

# TODO: Print the success boolean, result type, and result to inspect what the method returns

# Test case 2: Invalid handoff to non-existent agent
print("\n=== Test 2: Invalid Handoff (Agent Not Found) ===")
mock_tool_use_invalid = MockToolUse(
    name="handoff",
    input_dict={"name": "nonexistent_agent", "reason": "Testing error handling"},
    id="test_handoff_2"
)

# TODO: Call helpful_assistant.call_handoff(mock_tool_use_invalid, messages) with the invalid mock object

# TODO: Print the success boolean and result to see how the method handles errors
```

### File: `agent.py`

```python
import json
import anthropic


class Agent:
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        handoffs=None,
        max_turns=10
    ):
        self.client = anthropic.Anthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns

        # Avoid shared mutable defaults and protect against external mutation
        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        self.handoffs = [] if handoffs is None else list(handoffs)
        
        # Define handoff tool schema
        self.handoff_schema = {
            "name": "handoff",
            "description": "Transfer control to another specialized agent. Use this when the user's request is better handled by a different agent.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": f"Name of the agent to handoff to. Available agents: {[agent.name for agent in self.handoffs]}"
                    },
                    "reason": {
                        "type": "string", 
                        "description": "Brief explanation of why this handoff is needed"
                    }
                },
                "required": ["name", "reason"]
            }
        }
        
    def _extract_text(self, content):
        return "".join(
            block.text for block in content
            if getattr(block, "type", None) == "text"
        )    

    def _build_request_args(self, messages):
        request_args = {
            "model": self.model,
            "system": self.system_prompt,
            "messages": messages,
            "max_tokens": 8000,
        }

        # Build the complete tool schemas list
        all_tools = []
        
        # Add regular tool schemas if they exist
        if self.tool_schemas:
            all_tools.extend(self.tool_schemas)
        
        # Add handoff schema if handoffs are available
        if self.handoffs:
            all_tools.append(self.handoff_schema)
        
        # Add tools to request if any exist
        if all_tools:
            request_args["tools"] = all_tools

        return request_args

    def call_tool(self, tool_use):
        tool_name = tool_use.name
        tool_input = tool_use.input or {}
        tool_use_id = tool_use.id

        print(f"🔧 Tool called: {tool_name}({tool_input})")
        
        try:
            result = str(self.tools[tool_name](**tool_input))
        except KeyError:
            result = f"Error: Tool {tool_name} not found"
        except Exception as e:
            result = f"Error: {str(e)}"

        return {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result
        }
    
    def call_handoff(self, tool_use, messages):
        # TODO: Extract the agent name from tool_use.input using .get("name")
        agent_name = tool_use.input.get("name")
        
        # TODO: Extract the reason from tool_use.input using .get("reason", "No reason provided")
        reason = tool_use.input.get("reason", "No reason provided")
        
        # TODO: Print a debug message with the target agent name and handoff reason
        print(f"🔄 Handoff requested to agent '{agent_name}'. Reason: {reason}")
        
        # TODO: Start a try-except block to handle potential errors
        try:
            # TODO: Use next() with a generator expression to find the target agent in self.handoffs where agent.name matches agent_name
            target_agent = next(agent for agent in self.handoffs if agent.name == agent_name)
            
            # TODO: Clean the messages by removing the last assistant message if it exists 
            clean_messages = messages[:-1] if messages and messages[-1]["role"] == "assistant" else messages
            
            # TODO: Call target_agent.run(clean_messages) and return a tuple: (True, result)
            result = target_agent.run(clean_messages)
            return (True, result)
            
        # TODO: Add except StopIteration block to catch when agent is not found
        except StopIteration:
            # TODO: Return (False, tool_result_dict) with an error message
            error_msg = f"Error: Agent '{agent_name}' not found in available handoffs."
            tool_result_dict = {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": error_msg
            }
            return (False, tool_result_dict)
            
        # TODO: Add except Exception as e block to catch any other errors
        except Exception as e:
            # TODO: Return (False, tool_result_dict) with an error message
            tool_result_dict = {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Error during handoff: {str(e)}"
            }
            return (False, tool_result_dict)

    def run(self, input_messages):
        # Create a copy of the input messages to avoid modifying the original
        messages = input_messages.copy()

        # Loop until the model returns a final answer or the max turns is reached
        turn = 0
        while turn < self.max_turns:
            turn += 1

            response = self.client.messages.create(**self._build_request_args(messages))

            messages.append({"role": "assistant", "content": response.content})

            # Execute all tools if Claude requests any
            if response.stop_reason == "tool_use":
                tool_results = []
                for content_item in response.content:
                    if content_item.type == "tool_use":
                        tool_result = self.call_tool(content_item)
                        tool_results.append(tool_result)

                # Add all tool results to messages
                messages.append({
                    "role": "user",
                    "content": tool_results
                })
        
            else:
                # Return if no tools are requested
                response_text = self._extract_text(response.content)

                return messages, response_text

        # If the max turns is reached, raise an exception
        raise Exception("Max turns reached")

```

---

### File: `main.py`

```python
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load tool schemas
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Math tools
math_tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}

# Create a calculator assistant
calculator_assistant = Agent(
    name="calculator_assistant",
    system_prompt="You are a calculator assistant. You specialize in mathematical calculations and solving equations.",
    tools=math_tools,
    tool_schemas=tool_schemas
)

# Create a general assistant (can handoff to calculator)
helpful_assistant = Agent(
    name="helpful_assistant",
    system_prompt="You are a helpful assistant. You can assist with various tasks and handoff to the calculator assistant for math problems.",
    handoffs=[calculator_assistant]
)

# Mock class to simulate a tool_use object
class MockToolUse:
    def __init__(self, name, input_dict, id):
        self.name = name
        self.input = input_dict
        self.id = id

# Test case 1: Valid handoff to calculator_assistant
print("=== Test 1: Valid Handoff ===")
mock_tool_use = MockToolUse(
    name="handoff",
    input_dict={"name": "calculator_assistant", "reason": "Testing handoff logic"},
    id="test_handoff_1"
)

# Test messages with user and assistant messages
messages = [
    {"role": "user", "content": "Can you solve 2 + 2?"},
    {"role": "assistant", "content": [{"type": "text", "text": "Let me handoff to the calculator."}]}
]

# TODO: Call helpful_assistant.call_handoff(mock_tool_use, messages) and store the result in two variables: success and result
success, result = helpful_assistant.call_handoff(mock_tool_use, messages)

# TODO: Print the success boolean, result type, and result to inspect what the method returns
print(f"Handoff Success Status: {success}")
print(f"Result Type: {type(result)}")
print(f"Returned Result: {result}")


# Test case 2: Invalid handoff to non-existent agent
print("\n=== Test 2: Invalid Handoff (Agent Not Found) ===")
mock_tool_use_invalid = MockToolUse(
    name="handoff",
    input_dict={"name": "nonexistent_agent", "reason": "Testing error handling"},
    id="test_handoff_2"
)

# TODO: Call helpful_assistant.call_handoff(mock_tool_use_invalid, messages) with the invalid mock object
success_invalid, result_invalid = helpful_assistant.call_handoff(mock_tool_use_invalid, messages)

# TODO: Print the success boolean and result to see how the method handles errors
print(f"Handoff Success Status: {success_invalid}")
print(f"Error Result JSON Payloads:\n{json.dumps(result_invalid, indent=4)}")

```

## Detecting Handoffs and Transferring Control

Now that your Agent class has all the handoff infrastructure in place, it's time to wire up the execution logic that makes control transfers actually work. The run method currently processes all tool calls the same way, but handoffs need special treatment — they should immediately end the current agent's execution and return the target agent's response.

Your task is to modify the tool execution loop in the run method to detect and handle handoff tool calls differently from regular tools. When processing tool calls, you need to:

    Check if the tool name is "handoff" before processing it as a regular tool.
    Call self.call_handoff(content_item, messages) for handoff tools and capture the returned tuple as handoff_success, handoff_result.
    Immediately return handoff_result if handoff_success is True (this ends the current agent's involvement).
    Append handoff_result to tool_results if handoff_success is False (treating failed handoffs as regular tool results).
    Only execute regular tool processing for non-handoff tools.

Once you've implemented the handoff detection and handling, run main.py to test both scenarios: the helpful assistant should answer the France question directly, but hand off the linear equation to the calculator assistant, who will take complete control and solve it.

```
# agent.py
import json
import anthropic


class Agent:
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        handoffs=None,
        max_turns=10
    ):
        self.client = anthropic.Anthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns

        # Avoid shared mutable defaults and protect against external mutation
        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        self.handoffs = [] if handoffs is None else list(handoffs)  # list of other agents to handoff the control
        
        # Define handoff tool schema
        self.handoff_schema = {
            "name": "handoff",
            "description": "Transfer control to another specialized agent. Use this when the user's request is better handled by a different agent.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": f"Name of the agent to handoff to. Available agents: {[agent.name for agent in self.handoffs]}"
                    },
                    "reason": {
                        "type": "string", 
                        "description": "Brief explanation of why this handoff is needed"
                    }
                },
                "required": ["name", "reason"]
            }
        }
        
    def _extract_text(self, content):
        return "".join(
            block.text for block in content
            if getattr(block, "type", None) == "text"
        )    

    def _build_request_args(self, messages):
        request_args = {
            "model": self.model,
            "system": self.system_prompt,
            "messages": messages,
            "max_tokens": 8000,
        }

        # Build the complete tool schemas list
        all_tools = []
        
        # Add regular tool schemas if they exist
        if self.tool_schemas:
            all_tools.extend(self.tool_schemas)
        
        # Add handoff schema if handoffs are available
        if self.handoffs:
            all_tools.append(self.handoff_schema)
        
        # Add tools to request if any exist
        if all_tools:
            request_args["tools"] = all_tools

        # Return the complete set of arguments to use for the API call
        return request_args

    def call_tool(self, tool_use):
        tool_name = tool_use.name
        tool_input = tool_use.input or {}
        tool_use_id = tool_use.id

        print(f"🔧 Tool called: {tool_name}({tool_input})")
        
        try:
            result = str(self.tools[tool_name](**tool_input))
        except KeyError:
            result = f"Error: Tool {tool_name} not found"
        except Exception as e:
            result = f"Error: {str(e)}"

        return {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result
        }
        
    def call_handoff(self, tool_use, messages):
        # Extract the agent name from the handoff tool input
        agent_name = tool_use.input.get("name")
        reason = tool_use.input.get("reason", "No reason provided")
        
        print(f"🔄 Handoff to: {agent_name}")
        print(f"📝 Reason: {reason}")
        
        try:
            # Find the agent with the given name (raises StopIteration if not found)
            target_agent = next(agent for agent in self.handoffs if agent.name == agent_name)
            
            # Remove the last assistant message that contains the handoff tool_use
            clean_messages = messages[:-1] if messages and messages[-1]["role"] == "assistant" else messages
            
            # Handoff the control to the other agent
            return True, target_agent.run(clean_messages)
            
        except StopIteration:
            # Agent not found
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Agent '{agent_name}' not found. Available agents: {[agent.name for agent in self.handoffs]}"
            }
        except Exception as e:
            # Any other error during handoff execution
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Error during handoff to '{agent_name}': {str(e)}"
            }

    def run(self, input_messages):
        # Create a copy of the input messages to avoid modifying the original
        messages = input_messages.copy()

        # Loop until the model returns a final answer or the max turns is reached
        turn = 0
        while turn < self.max_turns:
            turn += 1

            response = self.client.messages.create(**self._build_request_args(messages))

            messages.append({"role": "assistant", "content": response.content})

            # Execute all tools if Claude requests any
            if response.stop_reason == "tool_use":
                tool_results = []
                for content_item in response.content:
                    if content_item.type == "tool_use":
                        # TODO: Check if this is a handoff tool call (content_item.name == "handoff")
                        # TODO: If it's a handoff, call self.call_handoff(content_item, messages)
                        # TODO: Store the result as: handoff_success, handoff_result
                        # TODO: If handoff_success is True, immediately return handoff_result
                        # TODO: If handoff_success is False, append handoff_result to tool_results
                        
                        # Execute regular tools
                        tool_result = self.call_tool(content_item)
                        # Add result to tool results list
                        tool_results.append(tool_result)

                # Add all tool results to messages
                messages.append({
                    "role": "user",
                    "content": tool_results
                })
        
            else:
                # Return if no tools are requested
                response_text = self._extract_text(response.content)

                return messages, response_text

        # If the max turns is reached, raise an exception
        raise Exception("Max turns reached")

# main.py
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load tool schemas
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Math tools
math_tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}

# Create a calculator assistant
calculator_assistant = Agent(
    name="calculator_assistant",
    system_prompt="You are a calculator assistant. You specialize in mathematical calculations and solving equations.",
    tools=math_tools,
    tool_schemas=tool_schemas,
    handoffs=[]
)

# Create a general assistant (can handoff to calculator)
helpful_assistant = Agent(
    name="helpful_assistant",
    system_prompt="You are a helpful assistant. You can assist with various tasks and handoff to the calculator assistant for math problems.",
    tools={},
    tool_schemas=[],
    handoffs=[calculator_assistant]
)

# Example test messages
messages1 = [
    {
        'role': 'user', 
        'content': 'What is the capital of France?'
    }
]

messages2 = [
    {
        'role': 'user', 
        'content': 'What is the solution to the equation 4x - 7 = 9?'
    }
]

# TODO: Call helpful_assistant.run with messages1 and print the response

# TODO: Call helpful_assistant.run with messages2 and print the response
```
Here is the complete implementation of both `agent.py` and `main.py` with the handoff detection and execution logic fully integrated into the tool execution loop.

### File: `agent.py`

```python
import json
import anthropic


class Agent:
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        handoffs=None,
        max_turns=10
    ):
        self.client = anthropic.Anthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns

        # Avoid shared mutable defaults and protect against external mutation
        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        self.handoffs = [] if handoffs is None else list(handoffs)  # list of other agents to handoff the control
        
        # Define handoff tool schema
        self.handoff_schema = {
            "name": "handoff",
            "description": "Transfer control to another specialized agent. Use this when the user's request is better handled by a different agent.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": f"Name of the agent to handoff to. Available agents: {[agent.name for agent in self.handoffs]}"
                    },
                    "reason": {
                        "type": "string", 
                        "description": "Brief explanation of why this handoff is needed"
                    }
                },
                "required": ["name", "reason"]
            }
        }
        
    def _extract_text(self, content):
        return "".join(
            block.text for block in content
            if getattr(block, "type", None) == "text"
        )    

    def _build_request_args(self, messages):
        request_args = {
            "model": self.model,
            "system": self.system_prompt,
            "messages": messages,
            "max_tokens": 8000,
        }

        # Build the complete tool schemas list
        all_tools = []
        
        # Add regular tool schemas if they exist
        if self.tool_schemas:
            all_tools.extend(self.tool_schemas)
        
        # Add handoff schema if handoffs are available
        if self.handoffs:
            all_tools.append(self.handoff_schema)
        
        # Add tools to request if any exist
        if all_tools:
            request_args["tools"] = all_tools

        # Return the complete set of arguments to use for the API call
        return request_args

    def call_tool(self, tool_use):
        tool_name = tool_use.name
        tool_input = tool_use.input or {}
        tool_use_id = tool_use.id

        print(f"🔧 Tool called: {tool_name}({tool_input})")
        
        try:
            result = str(self.tools[tool_name](**tool_input))
        except KeyError:
            result = f"Error: Tool {tool_name} not found"
        except Exception as e:
            result = f"Error: {str(e)}"

        return {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result
        }
        
    def call_handoff(self, tool_use, messages):
        # Extract the agent name from the handoff tool input
        agent_name = tool_use.input.get("name")
        reason = tool_use.input.get("reason", "No reason provided")
        
        print(f"🔄 Handoff to: {agent_name}")
        print(f"📝 Reason: {reason}")
        
        try:
            # Find the agent with the given name (raises StopIteration if not found)
            target_agent = next(agent for agent in self.handoffs if agent.name == agent_name)
            
            # Remove the last assistant message that contains the handoff tool_use
            clean_messages = messages[:-1] if messages and messages[-1]["role"] == "assistant" else messages
            
            # Handoff the control to the other agent
            return True, target_agent.run(clean_messages)
            
        except StopIteration:
            # Agent not found
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Agent '{agent_name}' not found. Available agents: {[agent.name for agent in self.handoffs]}"
            }
        except Exception as e:
            # Any other error during handoff execution
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Error during handoff to '{agent_name}': {str(e)}"
            }

    def run(self, input_messages):
        # Create a copy of the input messages to avoid modifying the original
        messages = input_messages.copy()

        # Loop until the model returns a final answer or the max turns is reached
        turn = 0
        while turn < self.max_turns:
            turn += 1

            response = self.client.messages.create(**self._build_request_args(messages))

            messages.append({"role": "assistant", "content": response.content})

            # Execute all tools if Claude requests any
            if response.stop_reason == "tool_use":
                tool_results = []
                for content_item in response.content:
                    if content_item.type == "tool_use":
                        # TODO: Check if this is a handoff tool call (content_item.name == "handoff")
                        if content_item.name == "handoff":
                            # TODO: If it's a handoff, call self.call_handoff(content_item, messages)
                            # TODO: Store the result as: handoff_success, handoff_result
                            handoff_success, handoff_result = self.call_handoff(content_item, messages)
                            
                            # TODO: If handoff_success is True, immediately return handoff_result
                            if handoff_success:
                                return handoff_result
                            # TODO: If handoff_success is False, append handoff_result to tool_results
                            else:
                                tool_results.append(handoff_result)
                                continue  # Skip processing it as a regular tool

                        # Execute regular tools
                        tool_result = self.call_tool(content_item)
                        # Add result to tool results list
                        tool_results.append(tool_result)

                # Add all tool results to messages
                messages.append({
                    "role": "user",
                    "content": tool_results
                })
        
            else:
                # Return if no tools are requested
                response_text = self._extract_text(response.content)

                return messages, response_text

        # If the max turns is reached, raise an exception
        raise Exception("Max turns reached")

```

---

### File: `main.py`

```python
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load tool schemas
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Math tools
math_tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}

# Create a calculator assistant
calculator_assistant = Agent(
    name="calculator_assistant",
    system_prompt="You are a calculator assistant. You specialize in mathematical calculations and solving equations.",
    tools=math_tools,
    tool_schemas=tool_schemas,
    handoffs=[]
)

# Create a general assistant (can handoff to calculator)
helpful_assistant = Agent(
    name="helpful_assistant",
    system_prompt="You are a helpful assistant. You can assist with various tasks and handoff to the calculator assistant for math problems.",
    tools={},
    tool_schemas=[],
    handoffs=[calculator_assistant]
)

# Example test messages
messages1 = [
    {
        'role': 'user', 
        'content': 'What is the capital of France?'
    }
]

messages2 = [
    {
        'role': 'user', 
        'content': 'What is the solution to the equation 4x - 7 = 9?'
    }
]

# TODO: Call helpful_assistant.run with messages1 and print the response
print("=== Scenario 1: General Knowledge Query ===")
_, response1 = helpful_assistant.run(messages1)
print(f"Final Response:\n{response1}\n")

# TODO: Call helpful_assistant.run with messages2 and print the response
print("=== Scenario 2: Math Task Handoff Routing ===")
_, response2 = helpful_assistant.run(messages2)
print(f"\nFinal Response:\n{response2}")

```